# Warehouse Intelligence: Analyst Walkthrough
This notebook loads the cleaned SQLite model, checks the data grain, then works through the evidence behind each recommendation in the management brief.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "warehouse_operations.sqlite").exists() else Path.cwd().parent
con = sqlite3.connect(ROOT / "warehouse_operations.sqlite")
picks = pd.read_sql_query("SELECT * FROM v_picks_enriched", con)
counts = pd.read_sql_query("SELECT * FROM cycle_counts_clean", con)
len(picks),len(counts)

(31830, 2573)

## 1. Validate the grain and source
Pick events and cycle counts have different grains, so each is summarised before any comparison.

In [2]:
assert picks.event_id.is_unique
print("Pick records:",len(picks))
print("Date coverage:",picks.pick_date.min(),"to",picks.pick_date.max())
print("Null duration rows:",picks.duration_seconds.isna().sum())

Pick records: 31830
Date coverage: 2026-01-05 to 2026-10-31
Null duration rows: 0


## 2. Describe operational performance
Rates use valid denominators. The per-pick benchmark is analyst-defined rather than a business SLA.

In [3]:
zone = picks.groupby("zone").agg(events=("event_id","size"),slow=("slow_pick","mean"),mismatches=("location_mismatch","mean"),gap_seconds=("benchmark_gap_seconds","sum")).reset_index()
zone["gap_hours"] = zone.gap_seconds/3600
zone.sort_values("gap_hours",ascending=False)

,zone,events,slow,mismatches,gap_seconds,gap_hours
2,Bulk,6880,0.342442,0.023547,151082.9,41.967472
1,Ambient-B,8570,0.256593,0.013186,131214.9,36.448583
0,Ambient-A,6961,0.249389,0.011205,103039.3,28.622028
3,Chilled,6388,0.270507,0.014089,100024.9,27.784694
4,High-Bay,3031,0.419993,0.034972,81743.5,22.706528


## 3. Stock-count error rates
Exact match is per counted event. It is not the percentage of SKUs fully accurate in the warehouse.

In [4]:
stock = counts.groupby("zone").agg(counts=("count_id","size"),match_rate=("exact_match","mean"),absolute_variance=("absolute_variance_units","sum"))
stock.sort_values("match_rate")

,counts,match_rate,absolute_variance
zone,,,
High-Bay,253,0.770751,354
Bulk,486,0.829218,528
Ambient-A,659,0.890744,462
Chilled,516,0.901163,316
Ambient-B,659,0.908953,378


## 4. Check confounding before recommendations
Breakdown and non-breakdown picks are compared within each zone so zone differences do not distort the result.

In [5]:
picks.groupby(["zone","pallet_breakdown"]).duration_seconds.agg(["count","mean","median"]).round(1)

count   mean  median
zone      pallet_breakdown                      
Ambient-A 0                  6093   68.9    65.0
          1                   868  135.5   131.0
Ambient-B 0                  7445   68.8    64.0
          1                  1125  136.4   131.0
Bulk      0                  5336   69.3    64.0
          1                  1544  136.5   130.0
Chilled   0                  5610   69.1    64.0
          1                   778  136.9   132.0
High-Bay  0                  2440   82.6    76.0
          1                   591  149.8   143.0

## 4b. Is the slow-pick KPI really measuring breakdowns?
The benchmark has no allowance for pallet breakdown, which adds roughly a minute. Check how much of the slow-pick rate it explains.

In [6]:
share = picks.loc[picks.slow_pick==1,"pallet_breakdown"].mean()
print(f"Slow picks involving a breakdown: {share:.1%}")
print(f"Slow-pick rate with breakdowns:    {picks.slow_pick.mean():.1%}")
print(f"Slow-pick rate without breakdowns: {picks.loc[picks.pallet_breakdown==0,'slow_pick'].mean():.1%}")

Slow picks involving a breakdown: 52.8%
Slow-pick rate with breakdowns:    29.2%
Slow-pick rate without breakdowns: 16.3%


## 4c. Totals versus rates
Total benchmark-gap hours reward busy zones. Gap seconds per pick shows intensity.

In [7]:
zone["gap_sec_per_pick"] = zone.gap_seconds/zone.events
zone.sort_values("gap_sec_per_pick",ascending=False).round(3)

,zone,events,slow,mismatches,gap_seconds,gap_hours,gap_sec_per_pick
4,High-Bay,3031,0.420,0.035,81743.5,22.707,26.969
2,Bulk,6880,0.342,0.024,151082.9,41.967,21.960
3,Chilled,6388,0.271,0.014,100024.9,27.785,15.658
1,Ambient-B,8570,0.257,0.013,131214.9,36.449,15.311
0,Ambient-A,6961,0.249,0.011,103039.3,28.622,14.802


## 4d. Shift comparison, like for like
Split by breakdown so Late's higher breakdown share cannot explain the whole gap. Then check the mix is otherwise similar.

In [8]:
display(picks.assign(sec_vs_target=picks.duration_seconds-picks.target_seconds)
        .groupby(["pallet_breakdown","shift"]).agg(picks=("event_id","size"),mean_sec=("duration_seconds","mean"),sec_vs_target=("sec_vs_target","mean")).round(1))
picks.groupby("shift").agg(breakdown_share=("pallet_breakdown","mean"),mean_qty=("qty_requested","mean"),
                          high_bay_share=("zone",lambda s:(s=="High-Bay").mean())).round(3)

picks  mean_sec  sec_vs_target
pallet_breakdown shift                                
0                Early  15475      67.3            0.0
                 Late   11449      74.2            7.1
1                Early   2454     134.4           67.2
                 Late    2452     141.6           74.4

,breakdown_share,mean_qty,high_bay_share
shift,,,
Early,0.137,3.984,0.095
Late,0.176,3.968,0.095


## 4e. Volume before bank holidays
Compare picks per operating day, not per pick: a pick-level count only reflects how many days are flagged.

In [9]:
daily = picks[picks.is_bank_holiday==0].groupby("pick_date").agg(picks=("event_id","size"),pre=("is_preholiday_5d","max"),dow=("day_of_week","first"))
display(daily.groupby("pre").picks.agg(["count","mean"]).round(1).rename(index={0:"other days",1:"5 days before holiday"}))
daily[daily.pre==0].groupby("dow").picks.mean().sort_values(ascending=False).round(1)

,count,mean
pre,,
other days,236,119.0
5 days before holiday,17,181.5


dow
Friday       145.4
Monday       144.4
Wednesday    115.9
Thursday     113.8
Tuesday      112.2
Saturday      83.5
Name: picks, dtype: float64

## 4f. Combined recount priority (SQL query 08)
Aggregate each source to one row per SKU before joining, then rank on rates.

In [10]:
pd.read_csv(ROOT/"data/processed/sku_recount_priority.csv")

,zone,sku_id,picks,counts,mismatch_pct,short_pick_pct,count_discrepancy_pct,priority_score
0,High-Bay,SKU0098,75,8,5.33,5.33,37.5,0.972
1,High-Bay,SKU0123,174,13,4.60,6.90,30.8,0.966
2,High-Bay,SKU0085,67,11,7.46,4.48,27.3,0.926
3,High-Bay,SKU0149,382,11,3.14,4.19,54.5,0.898
4,High-Bay,SKU0101,161,16,3.11,6.21,25.0,0.888
5,High-Bay,SKU0074,165,17,3.64,4.24,29.4,0.886
6,Chilled,SKU0110,72,15,4.17,4.17,26.7,0.884
7,High-Bay,SKU0139,171,19,6.43,2.92,36.8,0.880
8,High-Bay,SKU0070,159,13,4.40,3.14,23.1,0.843
9,Bulk,SKU0071,176,17,2.84,3.41,29.4,0.839


## 5. Conclusion & Reflection

This project allowed me to apply data analysis techniques to a realistic warehouse operations scenario, focusing on picking efficiency, inventory accuracy and operational performance. Using Python, SQL and Excel, I worked through the full analytical process, from data cleaning and validation to identifying patterns, visualising results and developing business recommendations.

The analysis covered 31,830 validated picking events and revealed several areas worth investigating. Although the overall unit fill rate was high at 98.93%, approximately 29.2% of picking events exceeded the defined time benchmark. High-Bay and Bulk zones showed particularly high rates of slower picks, suggesting opportunities to review stock positioning, inventory accuracy and pallet-handling processes.

One of the most valuable aspects of this project was learning to interpret findings within a business context. Rather than simply reporting performance figures, I focused on understanding what they could mean for warehouse operations and how management could use the information to make informed decisions. I also recognised the importance of distinguishing potential inefficiencies from proven causes, particularly when working with simulated data.

After completing the analysis, I ran the scripts locally, reviewed the outputs and verified that the different components worked together as intended. This strengthened my practical understanding of data preparation, SQL querying, automated validation and reporting.

Overall, this project reinforced my understanding that effective data analysis is not just about producing accurate calculations or visually appealing dashboards. It is about asking the right questions, ensuring the data is reliable and translating findings into practical recommendations that support better business decisions.

As a next step, I would like to develop the project further by introducing more advanced performance monitoring, interactive dashboards and additional data sources to explore how operational efficiency changes over time.